# Minimum Atomworks Output Analysis

This notebook reads `pdb.parquet` and `dataset.parquet` directly from a run output directory, summarizes the contents, and makes a few quick matplotlib plots for structure-, interface-, and dataset-level inspection.

Edit `OUT_DIR` in the next cell if you want to force a specific run directory.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from minimum_atw.core.output_files import dataset_output_path, pdb_output_path, read_output_metadata

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

REPO_ROOT = Path('/home/eva/minimum_atomworks')
OUT_DIR = None


def detect_out_dir(root: Path) -> Path:
    candidates = []
    for path in root.glob('out*'):
        if not path.is_dir():
            continue
        metadata = read_output_metadata(path)
        if pdb_output_path(path, metadata=metadata).exists():
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f'No output directory with a configured PDB parquet found under {root}')
    return max(candidates, key=lambda item: item.stat().st_mtime)


out_dir = Path(OUT_DIR).expanduser().resolve() if OUT_DIR else detect_out_dir(REPO_ROOT)
resolved_metadata = read_output_metadata(out_dir)
pdb_path = pdb_output_path(out_dir, metadata=resolved_metadata)
dataset_path = dataset_output_path(out_dir, metadata=resolved_metadata)
run_metadata_path = out_dir / 'run_metadata.json'
dataset_metadata_path = out_dir / 'dataset_metadata.json'

print(f'Using out_dir: {out_dir}')
print(f'PDB parquet path: {pdb_path.name} (exists={pdb_path.exists()})')
print(f'Dataset parquet path: {dataset_path.name} (exists={dataset_path.exists()})')
print(f'run_metadata.json exists: {run_metadata_path.exists()}')
print(f'dataset_metadata.json exists: {dataset_metadata_path.exists()}')

In [ ]:
IDENTITY_COLS = ['path', 'assembly_id', 'grain', 'chain_id', 'role', 'pair', 'role_left', 'role_right']


def read_json_if_exists(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def plugin_prefix(column: str) -> str:
    if '__' in column:
        return column.split('__', 1)[0]
    return 'identity'


def top_numeric_columns(frame: pd.DataFrame, *, skip: set[str], limit: int = 6) -> list[str]:
    numeric_cols = [
        col for col in frame.columns
        if col not in skip and pd.api.types.is_numeric_dtype(frame[col])
    ]
    numeric_cols.sort(key=lambda col: (frame[col].notna().sum(), col), reverse=True)
    return numeric_cols[:limit]


pdb = pd.read_parquet(pdb_path)
dataset = pd.read_parquet(dataset_path) if dataset_path.exists() else pd.DataFrame()
run_metadata = read_json_if_exists(run_metadata_path)
dataset_metadata = read_json_if_exists(dataset_metadata_path)

pdb_metric_cols = [col for col in pdb.columns if col not in IDENTITY_COLS]
dataset_identity_cols = ['analysis']
dataset_metric_cols = [col for col in dataset.columns if col not in dataset_identity_cols] if not dataset.empty else []

print('pdb rows:', len(pdb))
print('pdb columns:', len(pdb.columns))
if not dataset.empty:
    print('dataset rows:', len(dataset))
    print('dataset columns:', len(dataset.columns))
else:
    print('dataset rows: 0 (configured dataset parquet missing or empty)')

In [ ]:
grain_counts = pdb['grain'].value_counts().rename_axis('grain').reset_index(name='n_rows')
grain_metric_coverage = (
    pdb.groupby('grain')[pdb_metric_cols]
    .apply(lambda frame: frame.notna().sum())
    .transpose()
    if pdb_metric_cols else pd.DataFrame()
)

prefix_rows = []
for prefix in sorted({plugin_prefix(col) for col in pdb_metric_cols}):
    cols = [col for col in pdb_metric_cols if plugin_prefix(col) == prefix]
    if not cols:
        continue
    values = pdb[cols]
    prefix_rows.append({
        'prefix': prefix,
        'n_columns': len(cols),
        'rows_with_any_value': int(values.notna().any(axis=1).sum()),
        'non_null_cells': int(values.notna().sum().sum()),
    })
prefix_summary = pd.DataFrame(prefix_rows).sort_values(['rows_with_any_value', 'n_columns'], ascending=False)

print('Run metadata counts:', run_metadata.get('counts', {}))
print('Run metadata status summary:', run_metadata.get('status_summary', {}))
display(grain_counts)
display(prefix_summary)

if not grain_metric_coverage.empty:
    display(grain_metric_coverage.head(20))

if not dataset.empty:
    analysis_counts = dataset['analysis'].value_counts().rename_axis('analysis').reset_index(name='n_rows')
    print('Dataset metadata counts:', dataset_metadata.get('counts', {}))
    print('Dataset metadata status summary:', dataset_metadata.get('status_summary', {}))
    display(analysis_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(grain_counts['grain'], grain_counts['n_rows'], color='#1f77b4')
axes[0].set_title('PDB Rows by Grain')
axes[0].set_xlabel('grain')
axes[0].set_ylabel('rows')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(prefix_summary['prefix'], prefix_summary['rows_with_any_value'], color='#ff7f0e')
axes[1].set_title('Rows with Values by Metric Prefix')
axes[1].set_xlabel('prefix')
axes[1].set_ylabel('rows with any metric value')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
selected_pdb_numeric = top_numeric_columns(pdb, skip=set(IDENTITY_COLS), limit=6)
display(pdb[selected_pdb_numeric].describe().transpose() if selected_pdb_numeric else pd.DataFrame())

if selected_pdb_numeric:
    ncols = 2
    nrows = math.ceil(len(selected_pdb_numeric) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, column in zip(axes, selected_pdb_numeric):
        values = pdb[column].dropna()
        ax.hist(values, bins=min(30, max(5, len(values))), color='#2ca02c', edgecolor='black')
        ax.set_title(column)
        ax.set_xlabel('value')
        ax.set_ylabel('count')
    for ax in axes[len(selected_pdb_numeric):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric PDB metric columns found.')

In [ ]:
interface_rows = pdb.loc[pdb['grain'] == 'interface'].copy()
role_rows = pdb.loc[pdb['grain'] == 'role'].copy()

if not interface_rows.empty:
    top_interface_numeric = top_numeric_columns(interface_rows, skip=set(IDENTITY_COLS), limit=4)
    print('Top interface metric columns:', top_interface_numeric)
    display(interface_rows[['pair'] + top_interface_numeric].head(20) if top_interface_numeric else interface_rows[['pair']].head(20))

if not role_rows.empty and 'role' in role_rows.columns:
    role_counts = role_rows['role'].fillna('').replace('', '(blank)').value_counts().rename_axis('role').reset_index(name='n_rows')
    display(role_counts)
    plt.figure(figsize=(8, 4))
    plt.bar(role_counts['role'], role_counts['n_rows'], color='#9467bd')
    plt.title('Role Rows by Role Name')
    plt.xlabel('role')
    plt.ylabel('rows')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
if dataset.empty:
    print('Configured dataset parquet not present for this run.')
else:
    analysis_counts = dataset['analysis'].value_counts().rename_axis('analysis').reset_index(name='n_rows')
    dataset_numeric = top_numeric_columns(dataset, skip={'analysis'}, limit=6)
    display(dataset[dataset_numeric].describe().transpose() if dataset_numeric else pd.DataFrame())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(analysis_counts['analysis'], analysis_counts['n_rows'], color='#17becf')
    axes[0].set_title('Dataset Rows by Analysis')
    axes[0].set_xlabel('analysis')
    axes[0].set_ylabel('rows')
    axes[0].tick_params(axis='x', rotation=35)

    if dataset_numeric:
        dataset[dataset_numeric].notna().sum().sort_values(ascending=False).plot(
            kind='bar',
            ax=axes[1],
            color='#8c564b'
        )
        axes[1].set_title('Dataset Numeric Column Coverage')
        axes[1].set_xlabel('column')
        axes[1].set_ylabel('non-null rows')
        axes[1].tick_params(axis='x', rotation=60)
    else:
        axes[1].text(0.5, 0.5, 'No numeric dataset columns', ha='center', va='center')
        axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()